In [ ]:
import os
import numpy as np
import rasterio
from rasterio.enums import Resampling

# === Define input paths ===
base_dir = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\Forest_Edge_Area_2021_1km"
area_1988_path = os.path.join(base_dir, "Forest_Area_1988_1km.tif")
area_2021_path = os.path.join(base_dir, "Forest_Area_2021_1km.tif")
edge_1988_path = os.path.join(base_dir, "Edge_Length_1988_1km.tif")
edge_2021_path = os.path.join(base_dir, "Edge_Length_2021_1km.tif")

# === Define output paths ===
area_diff_path = os.path.join(base_dir, "Forest_Area_Diff_2021_minus_1988_1km.tif")
edge_diff_path = os.path.join(base_dir, "Edge_Length_Diff_2021_minus_1988_1km.tif")
rel_area_path = os.path.join(base_dir, "Forest_Area_RelDiff_2021_div_1988_1km.tif")
rel_edge_path = os.path.join(base_dir, "Edge_Length_RelDiff_2021_div_1988_1km.tif")

# === Common nodata value for float outputs ===
output_nodata = -9999.0

# === Load raster and mask nodata values ===
def load_raster(path):
    with rasterio.open(path) as src:
        data = src.read(1, resampling=Resampling.nearest).astype(np.float32)
        profile = src.profile
        nodata = src.nodata
        if nodata is not None:
            data[data == nodata] = np.nan
    return data, profile

# === Save raster with correct nodata handling ===
def save_raster(data, profile, out_path):
    profile.update(dtype=rasterio.float32, compress='lzw', nodata=output_nodata)
    data = np.where(np.isnan(data), output_nodata, data).astype(np.float32)
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(data, 1)

# === Load rasters ===
area_1988, profile = load_raster(area_1988_path)
area_2021, _ = load_raster(area_2021_path)
edge_1988, _ = load_raster(edge_1988_path)
edge_2021, _ = load_raster(edge_2021_path)

# === Calculate differences ===
area_diff = area_2021 - area_1988
edge_diff = edge_2021 - edge_1988

# === Calculate relative differences with divide safety ===
with np.errstate(divide='ignore', invalid='ignore'):
    rel_area = np.where(area_1988 != 0, area_diff / area_1988, np.nan)
    rel_edge = np.where(edge_1988 != 0, edge_diff / edge_1988, np.nan)

# === Mask out pixels where both 1988 and 2021 forest area are 0 ===
mask_invalid = (area_1988 == 0) & (area_2021 == 0)
area_diff[mask_invalid] = np.nan
edge_diff[mask_invalid] = np.nan
rel_area[mask_invalid] = np.nan
rel_edge[mask_invalid] = np.nan

# === Save output rasters ===
save_raster(area_diff, profile, area_diff_path)
save_raster(edge_diff, profile, edge_diff_path)
save_raster(rel_area, profile, rel_area_path)
save_raster(rel_edge, profile, rel_edge_path)

print("✅ All 4 TIFs saved with nodata handled correctly.")

In [ ]:
import math
import os
import numpy as np
import rasterio
from rasterio.enums import Resampling

base_dir = r"G:\Hangkai\CONUS_Forest_Edge_LCMAP\Forest_Edge_Area_2021_1km"
area_1988_path = os.path.join(base_dir, "Forest_Area_1988_1km.tif")
area_2021_path = os.path.join(base_dir, "Forest_Area_2021_1km.tif")
edge_1988_path = os.path.join(base_dir, "Edge_Length_1988_1km.tif")
edge_2021_path = os.path.join(base_dir, "Edge_Length_2021_1km.tif")

def summarize_changes(area_1988_path, area_2021_path, edge_1988_path, edge_2021_path):
    def sum_change_pair(p1, p2):
        total_change = 0.0
        baseline_sum = 0.0
        pos = neg = 0

        with rasterio.open(p1) as s1, rasterio.open(p2) as s2:
            assert s1.width == s2.width and s1.height == s2.height, "Size mismatch"
            nod1, nod2 = s1.nodata, s2.nodata

            for ji, window in s1.block_windows(1):
                a = s1.read(1, window=window).astype(np.float32)
                b = s2.read(1, window=window).astype(np.float32)

                if nod1 is not None:
                    a = np.where(a == nod1, np.nan, a)
                if nod2 is not None:
                    b = np.where(b == nod2, np.nan, b)

                diff = b - a
                # Mask pixels where both baseline and current are 0 (like your script)
                mask_invalid = (a == 0) & (b == 0)
                diff = np.where(mask_invalid, np.nan, diff)

                total_change += float(np.nansum(diff))
                baseline_sum += float(np.nansum(np.where(np.isnan(a), 0.0, a)))

                pos += int(np.sum(np.isfinite(diff) & (diff > 0)))
                neg += int(np.sum(np.isfinite(diff) & (diff < 0)))

        pct = 100.0 * total_change / baseline_sum if baseline_sum > 0 else np.nan
        return total_change, baseline_sum, pct, pos, neg

    area_change, area_base, area_pct, area_pos, area_neg = sum_change_pair(area_1988_path, area_2021_path)
    edge_change, edge_base, edge_pct, edge_pos, edge_neg = sum_change_pair(edge_1988_path, edge_2021_path)

    print("===== Forest Area (2021 - 1988) =====")
    print(f"Total change: {area_change:,.2f} km²")
    print(f"Baseline (1988): {area_base:,.2f} km²")
    print(f"Percent change (vs 1988): {area_pct:.3f}%")
    print(f"Pixels with increase: {area_pos:,} | decrease: {area_neg:,}")

    print("\n===== Edge Length (2021 - 1988) =====")
    print(f"Total change: {edge_change:,.2f} km")
    print(f"Baseline (1988): {edge_base:,.2f} km")
    print(f"Percent change (vs 1988): {edge_pct:.3f}%")
    print(f"Pixels with increase: {edge_pos:,} | decrease: {edge_neg:,}")

summarize_changes(area_1988_path, area_2021_path, edge_1988_path, edge_2021_path)
